# Feed Community Ground Truth

In [1]:
import sys

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["DGLBACKEND"] = "pytorch"

import torch

DEVICE = torch.device("cuda:0")  # GPU 1 becomes cuda:0 now

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
(12, 0)


In [2]:

import torch
from collections import defaultdict
import statistics as stats
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
 
from dotenv import load_dotenv
from huggingface_hub import login
 
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token=token)
 
# MODEL_ID = "Qwen/Qwen3-14B"
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
# MODEL_ID = "mistralai--Mistral-7B-v0.1"
# MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"   # gated -- must accept license + login() above
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",       # spreads layers across available GPU(s), offloads to CPU if needed
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded.")

/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0%|          | 2/579 [00:00<01:16,  7.53it/s]/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 579/579 [00:02<00:00, 194.29it/s]


Model loaded.


## 1. Load files

In [3]:

import os
import json
import re
import torch
import pandas as pd

  
ROI_NAMES_JSON = "../aal_roi_names.json"                # region-name legend, matches BLEG's "{template}" field
META_CSV = "../subject_summary.csv"


EDGE_LIST_NUMBERS = "../edge_list_top10pos_numbers.json"   # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}


EDGE_LIST_NAMES = "../edge_list_top10pos_names.json"  # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}
EDGE_LIST_NAMES_ONLY = "../edge_only_from_top10pos_names.json"

ZSCORED_NUMBERS = "../zscore_top10pos_numbers.json"  
ZSCORED_NAMES = "../zscore_top10pos_names.json" 
ROI_NAMES_JSON = "../aal_roi_names.json"                # region-name legend, matches BLEG's "{template}" field
META_CSV = "../subject_summary.csv"
DATASET_NAME = "ADHD-200"
LABEL1, LABEL2 = "Control", "ADHD"
HUB_GROUND_TRUTH = "../hub_ground_truth_20roi.json"  # {subject_id: "Control" or "ADHD"}
COMMUNITY_GROUND_TRUTH = "../community_ground_truth.json"
COMMUNITY_GROUND_TRUTH_COUNTS = "../community_ground_truth_counts.json"
ROI_NETWORK_JSON = "../roi_network.json"

PREPROCESS_TEMPLATE = "AAL116"
TASK_DESC = "interpreting functional connectivity patterns and reasoning about the most likely diagnostic group"
N_SUBJECTS = 768
MAX_NEW_TOKENS = 800

In [4]:
# with open(EDGE_LIST_NUMBERS) as f:
# with open(EDGE_LIST_NAMES) as f:
# with open(EDGE_LIST_NAMES_ONLY) as f:
with open(COMMUNITY_GROUND_TRUTH_COUNTS) as f:
# with open(CLINICAL_HUB_PERTURBED_K5) as f:
# with open(CLINICAL_HUB_PERTURBED_K10) as f:
# with open(HUB_CLINICAL_PERTURBED_NETWORK_K10) as f:
# with open(RANDOM_HUB_PERTURBED_K10) as f:
# with open(RANDOM_HUB_PERTURBED_K5) as f:
# with open(HUB_GROUND_TRUTH) as f:
# with open(ZSCORED_NUMBERS) as f:
# with open(ZSCORED_NAMES) as f:
    structure_prompts_all = json.load(f)

if os.path.exists(ROI_NAMES_JSON):
    with open(ROI_NAMES_JSON) as f:
        roi_names = json.load(f)
    print(f"Loaded {len(roi_names)} AAL116 ROI names")
else:
    roi_names = None
    print("[warn] aal_roi_names.json not found -- description will omit the ROI-name legend")

if os.path.exists(META_CSV):
    meta = pd.read_csv(META_CSV, dtype={"subject_id": str})
else:
    meta = None

subject_ids = list(structure_prompts_all.keys())[:N_SUBJECTS]
print(f"Loaded {len(structure_prompts_all)} subjects, running on first {len(subject_ids)}")
brain_graph_texts = dict(structure_prompts_all)
print("\nExample BrainGraph text (subject 1):")

print(brain_graph_texts[subject_ids[0]])

Loaded 116 AAL116 ROI names
Loaded 768 subjects, running on first 768

Example BrainGraph text (subject 1):
{'ground_truth': 'Control', 'within_network': {'Vis': 87, 'Default': 67, 'SomMot': 46, 'Unassigned': 39, 'Limbic': 25, 'Cont': 13, 'SalVentAttn': 12, 'DorsAttn': 1}, 'between_network': {'Default-Limbic': 74, 'SalVentAttn-SomMot': 32, 'Limbic-Vis': 26, 'Default-Unassigned': 22, 'Limbic-Unassigned': 22, 'Cont-SomMot': 21, 'DorsAttn-Vis': 17, 'Unassigned-Vis': 16, 'Cont-SalVentAttn': 15, 'Cont-Limbic': 12, 'Default-SalVentAttn': 12, 'Limbic-SalVentAttn': 12, 'SalVentAttn-Unassigned': 12, 'SomMot-Unassigned': 12, 'Cont-Unassigned': 10, 'Default-Vis': 10, 'Cont-Default': 9, 'SomMot-Vis': 8, 'Default-SomMot': 7, 'Limbic-SomMot': 7, 'DorsAttn-SomMot': 5, 'DorsAttn-Limbic': 4, 'Cont-DorsAttn': 3, 'Default-DorsAttn': 3, 'DorsAttn-Unassigned': 3, 'SalVentAttn-Vis': 2, 'Cont-Vis': 1, 'DorsAttn-SalVentAttn': 0}}


## 3. Prompt Template -- Description / Task / Request

In [5]:
def build_stage1_prompt(subject_id, community_data):
    within_str = "\n".join(f"  {net}: {count}" for net, count in community_data["within_network"].items())
    between_str = "\n".join(f"  {pair}: {count}" for pair, count in community_data["between_network"].items())

    return f"""Template: ADHD-200 dataset, AAL116 atlas, 116 ROIs.
Description: The following are within-network and between-network connectivity degree counts for the subject's resting-state functional connectivity graph, computed across the functional networks. Each within-network value represents the number of strong connections involving ROIs within that network, and each between-network value represents the number of strong connections between the two specified networks. "Unassigned" refers to ROIs (mainly cerebellum and vermis regions) that have no cortical overlap with any of the 7 standard functional networks, so they are grouped separately rather than forced into one.
Task: Based only on the network connectivity degree counts below, analyze the brain's network-level connectivity pattern and predict whether the subject belongs to Control or ADHD. Do not use or infer any ground-truth label from the input. Provide a confidence score (0-1) for both classes.
Request: Your output must strictly follow this JSON structure and contain nothing else:
{{
  "prediction": "Control or ADHD",
  "class_confidence": {{"Control": 0.0,"ADHD": 0.0}},
  "reasoning": "(1-2 sentence)"
}}
Network connectivity degree counts:
Within-network:
{within_str}

Between-network:
{between_str}"""

first_sid = subject_ids[0]
first_prompt = build_stage1_prompt(first_sid, brain_graph_texts[first_sid])
print(f"=== INPUT SANITY CHECK -- Subject {first_sid} ===")
print(first_prompt)
print(f"\n--- prompt length: {len(first_prompt)} chars, "
      f"{len(tokenizer(first_prompt)['input_ids'])} tokens ---")
input_tokens = tokenizer.encode(first_prompt)

total_input_tokens = len(input_tokens)

print(f"Total input tokens: {total_input_tokens}")


=== INPUT SANITY CHECK -- Subject 1018959 ===
Template: ADHD-200 dataset, AAL116 atlas, 116 ROIs.
Description: The following are within-network and between-network connectivity degree counts for the subject's resting-state functional connectivity graph, computed across the functional networks. Each within-network value represents the number of strong connections involving ROIs within that network, and each between-network value represents the number of strong connections between the two specified networks. "Unassigned" refers to ROIs (mainly cerebellum and vermis regions) that have no cortical overlap with any of the 7 standard functional networks, so they are grouped separately rather than forced into one.
Task: Based only on the network connectivity degree counts below, analyze the brain's network-level connectivity pattern and predict whether the subject belongs to Control or ADHD. Do not use or infer any ground-truth label from the input. Provide a confidence score (0-1) for both c

## 4. Generate -- one subject per call, human-readable parsed output

In [6]:
SAVE_EVERY = 10         
RETRY_ATTEMPTS = 2       

OUTPUT_DIR = "deepseek/community"

CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_community_gt_36_checkpoint_10com_1.json")

FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_community_gt_36_10com_1.json")

LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_community_gt_36_errors_10com_1.log")




In [ ]:
import json
import re

# def parse_json(raw):
#     raw = raw.strip()

#     # Remove reasoning
#     if "</think>" in raw:
#         raw = raw.split("</think>", 1)[1].strip()

#     # Remove ```json fences
#     raw = re.sub(r"```(?:json)?|```", "", raw).strip()

#     # Extract JSON object
#     start = raw.find("{")
#     end = raw.rfind("}")

#     if start == -1 or end == -1:
#         return None

#     try:
#         return json.loads(raw[start:end + 1])
#     except json.JSONDecodeError as e:
#         print("Parse error:", e)
#         print(raw)
#         return None


# ##-----------------------Qwen3--------------------------
import json

def parse_json(raw):
    raw = raw.strip()

    start = raw.find("{")
    if start == -1:
        return None

    raw = raw[start:]

    diff = raw.count("{") - raw.count("}")
    if diff > 0:
        raw += "}" * diff

    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(e)
        print(raw)
        return None


##-----------------------GPT and Deepseek and Deepseek-distill--------------------------

# def parse_json(raw_text):
#     # grab the reasoning trace, if the model produced one
#     think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
#     reasoning = think_match.group(1).strip() if think_match else None

#     match = re.search(r"\{.*\}", raw_text, re.DOTALL)
#     if not match:
#         return None
#     try:
#         parsed = json.loads(match.group(0))
#     except json.JSONDecodeError:
#         return None

#     if not isinstance(parsed, dict):
#         return None

#     if reasoning:
#         parsed["reasoning"] = reasoning

#     return parsed

In [ ]:

import atexit
import gc
import json
import os
import signal
import time

import torch
from tqdm.auto import tqdm



def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            loaded = json.load(f)
        # Only trust entries that actually completed successfully -- guards
        # against an old-format checkpoint that might contain error records.
        loaded = {sid: rec for sid, rec in loaded.items() if rec.get("prediction") is not None}
        print(f"Resuming: {len(loaded)} subjects already completed successfully.")
        return loaded
    return {}


def save_checkpoint(results, path=CHECKPOINT_PATH):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, path)  # atomic write


def log_line(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_PATH, "a") as f:
        f.write(f"[{ts}] {msg}\n")

##---------------------qwen3 and DeepSeek-R1-Distill-Qwen-14B----------------

def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_stage1_prompt(subject_id, brain_graph_text)
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    THINK_BUDGET = 150          # tokens allowed for reasoning
    ANSWER_BUDGET = max_new_tokens - THINK_BUDGET   # remaining ~300 for the JSON

    try:
        with torch.no_grad():
            # Stage 1: let it think, but cap it
            think_ids = model.generate(
                **inputs, max_new_tokens=THINK_BUDGET,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )
            think_text = tokenizer.decode(
                think_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )
            # force-close thinking if it didn't finish on its own
            if "</think>" not in think_text:
                think_text += "\n</think>\n"

            # Stage 2: continue from forced-closed think, now answer in JSON
            stage2_prompt = prompt_text + think_text
            inputs2 = tokenizer(stage2_prompt, return_tensors="pt").to(model.device)
            answer_ids = model.generate(
                **inputs2, max_new_tokens=ANSWER_BUDGET,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )
            answer_text = tokenizer.decode(
                answer_ids[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True
            )
        return think_text + answer_text
    finally:
        del inputs
        gc.collect()
        torch.cuda.empty_cache()




# def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS):
#     prompt = build_stage1_prompt(subject_id, brain_graph_text)
#     messages = [
#         {
#             "role": "system",
#             "content": (
#                 "You are a neuroscience expert. Output only one valid JSON object matching the required schema. No markdown, comments, or extra text. Must parse with json.loads()."
#             ),
#         },
#         {"role": "user", "content": prompt},
#     ]
#     prompt_text = tokenizer.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=True,
#     )
#     inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
#     try:
#         with torch.no_grad():
#             output_ids = model.generate(
#                 **inputs,
#                 max_new_tokens=max_new_tokens,
#                 do_sample=False,
#                 pad_token_id=tokenizer.eos_token_id,
#             )
#         generated = output_ids[0][inputs["input_ids"].shape[1]:]
#         text = tokenizer.decode(generated, skip_special_tokens=True).strip()
#         return text
#     finally:
        # # Runs whether generate() succeeded or raised -- keeps memory from
        # # accumulating across 800+ sequential calls.
        # del inputs
        # if "output_ids" in dir():
        #     del output_ids
        # gc.collect()
        # torch.cuda.empty_cache()


# #---------------------GPT and Deepseek-V4--------------------
# def generate_stage1_response_safe(subject_id, brain_graph_text, max_new_tokens=MAX_NEW_TOKENS, retries=2):
#     prompt = build_stage1_prompt(subject_id, brain_graph_text)  # unchanged prompt
#     for attempt in range(retries + 1):
#         try:
#             response = client.chat.completions.create(
#                 model=MODEL_ID,
#                 messages=[{"role": "user", "content": prompt}],
#                 max_completion_tokens=max_new_tokens,   # was max_tokens
#             )
#             return response.choices[0].message.content
#         except Exception:
#             if attempt == retries:
#                 raise
#             time.sleep(2 * (attempt + 1))

def process_one_subject(sid):
    """Returns a result dict on success, or raises the last exception after
    exhausting RETRY_ATTEMPTS."""
    last_exc = None
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
            parsed = parse_json(raw)

            gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
            ground_truth = "Control" if gt == 0 else "ADHD"

            if parsed is None:
                # Model ran fine but didn't return valid JSON -- this is NOT
                # a CUDA error, retrying won't help, log it and give up on
                # this subject for this run (still excluded from JSON).
                log_line(f"[parse_fail] {sid} (attempt {attempt}): {raw[:300]!r}")
                raise ValueError("Could not parse JSON from model output")

            return {
                "subject_id": sid,
                "ground_truth": ground_truth,
                "prediction": parsed.get("prediction", ""),
                "class_confidence": parsed.get("class_confidence", {}),
                "reasoning": parsed.get("reasoning", ""),
            }
        except Exception as e:
            last_exc = e
            log_line(f"[error] {sid} (attempt {attempt}/{RETRY_ATTEMPTS}): {repr(e)}")
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    raise last_exc

sid = subject_ids[15]  # or any specific ID, e.g. "1018959"

raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
print(repr(raw))  # exact raw text, for debugging if parsing fails

parsed = parse_json(raw)

if parsed is None:
    print("\nPARSE FAILED -- raw text above was not valid JSON.")
else:
    print("\nPARSED SUCCESSFULLY:")
    print(f"  prediction:       {parsed.get('prediction')}")
    print(f"  class_confidence: {parsed.get('class_confidence')}")
    print(f"  reasoning:        {parsed.get('reasoning')}")

/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


'Okay, so I\'m trying to figure out whether the subject belongs to the Control group or ADHD based on their functional connectivity graph. The data provided includes within-network and between-network connectivity counts using the AAL116 atlas and the ADHD-200 dataset. \n\nFirst, I\'ll look at the within-network connections. The highest within-network connectivity is in the Vis (Visual) network with 157 connections. That\'s a lot, so the visual network is very active. The next is Limbic with 53, which is also quite high. The Unassigned network has 44, which is lower than Vis but still significant. The others like SomMot, Default, Cont, SalVentAttn, and D\n</think>\n{\n  "prediction": "ADHD",\n  "class_confidence": {"Control": 0.3, "ADHD": 0.7},\n  "reasoning": "The high within-network connectivity in the Visual and Limbic networks, along with strong between-network connections like Limbic-Unassigned and Default-Limbic, suggests increased network segregation, which is often associated w

## Inference

In [9]:
# ---------------------------------------------------------------------
# Resume from checkpoint, figure out what's left to do
# ---------------------------------------------------------------------
results = load_checkpoint()
remaining_ids = [sid for sid in subject_ids if sid not in results]
print(f"Total subjects: {len(subject_ids)} | already done: {len(results)} | remaining: {len(remaining_ids)}")

n_success, n_parse_fail, n_error = 0, 0, 0
start_time = time.time()

with open(LOG_PATH, "a") as log_f:
    pbar = tqdm(remaining_ids, desc="Subjects", unit="subj")
    for sid in pbar:
        try:
            raw = generate_stage1_response_safe(sid, brain_graph_texts[sid])
            parsed = parse_json(raw)

            gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
            ground_truth = "Control" if gt == 0 else "ADHD"

            if parsed:
                results[sid] = {
                    "subject_id": sid,
                    "ground_truth": ground_truth,
                    "prediction": parsed.get("prediction", ""),
                    "class_confidence": parsed.get("class_confidence", {}),
                    "reasoning": parsed.get("reasoning", ""),
                    }
                n_success += 1
            else:
                # Model ran fine, just didn't return valid JSON -- this
                # counts as "done", NOT retried. Raw output is kept so
                # you can inspect/re-parse it later if needed.
                results[sid] = {
                    "subject_id": sid,
                    "ground_truth": ground_truth,
                    "raw_output": raw,
                    "parsed": None,
                }
                n_parse_fail += 1
                log_f.write(f"[parse_fail] {sid}: {raw[:300]!r}\n")
                log_f.flush()

        except Exception as e:
            # A real crash (CUDA error, etc.) -- NOT saved to results, stays
            # "not done", so this exact subject is retried the next time
            # this cell runs.
            n_error += 1
            log_f.write(f"[error] {sid}: {repr(e)}\n")
            log_f.flush()
            gc.collect()
            torch.cuda.empty_cache()

        done = len(results)
        pbar.set_postfix(done=done, ok=n_success, parse_fail=n_parse_fail, err=n_error)

        if done % SAVE_EVERY == 0:
            save_checkpoint(results)

# Final save + promote checkpoint to the "done" filename once everything ran
save_checkpoint(results)
elapsed = time.time() - start_time
print(f"\nFinished this run in {elapsed/60:.1f} min.")
print(f"Total saved: {len(results)}/{len(subject_ids)}  "
      f"(ok={n_success}, parse_fail={n_parse_fail}, error={n_error})")

if len(results) == len(subject_ids): 
    os.replace(CHECKPOINT_PATH, FINAL_PATH)
    print(f"All subjects done -> {FINAL_PATH}")
else:
    print(f"Not all subjects done yet -- re-run this cell to resume from {CHECKPOINT_PATH}")


Total subjects: 768 | already done: 0 | remaining: 768


Subjects:   1%|          | 4/768 [01:05<4:14:09, 19.96s/subj, done=4, err=0, ok=3, parse_fail=1]

Parse error: Expecting ',' delimiter: line 3 column 52 (char 77)
Alright, let's break this down step by step.

1. **Within-Network Analysis**:
   - The Visual (Vis) network has the highest within-network connectivity with 129 connections. This indicates a highly connected and potentially very active visual network, which is typical in control subjects as they often show strong visual processing.
   - The Limbic network has 53 connections, which is moderate. The Unassigned regions also have 53, suggesting they are functioning similarly to the Limbic network.
   - The Default network has 42 connections, which is lower than the Visual but still significant. Lower default mode activity is sometimes seen in ADHD, but here it's moderate.
   - The SomMot network has 33, Cont at 21, SalVentAttn at 7, and DorsAttn at 1. These lower values might indicate less connectivity in attention-related networks, which could be a sign of ADHD.

2. **Between-Network Analysis**:
   - The highest between-netw

Subjects:  58%|█████▊    | 442/768 [1:44:02<1:47:20, 19.76s/subj, done=442, err=0, ok=438, parse_fail=4]

Parse error: Expecting ',' delimiter: line 3 column 52 (char 77)
lowest at just 1 connection. This suggests that the DorsAttn network is not very active or connected within itself.

Next, I'll examine the between-network connections. The most significant connections are between Limbic and Unassigned with 52, and Default and Limbic with 49. This indicates strong cross-talk between the Limbic network and both the Default and Unassigned networks. The Unassigned network is also highly connected to Vis with 35 connections, which might mean that the cerebellum and vermis regions are interacting a lot with the visual areas.

Looking at the SalVentAttn and SomMot networks, their connections are mostly with other networks like Unassigned and each other. For example, SalVentAttn-SomMot has 24 connections, which is a moderate number. However, SalVentAttn is also connected to Unassigned with 11, and SomMot to Unassigned with 12. This might suggest that these attention and sensory-motor networks ar

...

Parse error: Expecting ',' delimiter: line 3 column 52 (char 77)
6, and DorsAttn 1. 

Looking at the between-network connections, the most significant connections are between Default and Limbic (34), Cont and Default (27), Limbic and Vis (21), Limbic and Unassigned (20), Default and Unassigned (18), Default and SalVentAttn (17), SalVentAttn and SomMot (17), Cont and Limbic (16), Limbic and SalVentAttn (14), DorsAttn and Vis (12), Limbic and SomMot (11), SomMot and Vis (11), Default and Vis (10), Unassigned and Vis (10), SalVentAttn and Unassigned (9), Cont and Unassigned (8), Cont and SalVentAttn (7), Cont and DorsAttn (6), SomMot and Unassigned (6), Cont and SomMot (5), Default and SomMot (5), DorsAttn and Limbic (4), DorsAttn and SomMot (3), Cont and Vis (2), Default and DorsAttn (0), DorsAttn and SalVentAttn (0), DorsAttn and Unassigned (0), SalVentAttn and Vis (0).

From this, I notice that the Default mode network is highly connected both within itself and with other networks like

Subjects:  75%|███████▍  | 573/768 [2:14:50<1:02:57, 19.37s/subj, done=573, err=0, ok=566, parse_fail=7]

Parse error: Expecting ',' delimiter: line 3 column 52 (char 80)
The within-network connections show strong connectivity in the Visual and Default networks, which are typically associated with sensory processing and the default mode network, respectively. The Default network is known to be involved in self-referential thought and is often more active in ADHD, but the high within-Default connectivity here might suggest more focused processing. However, the high within-Vis suggests strong integration within the visual network, which could be typical for controls.

Looking at between-network connections, the strongest is Default-Limbic with 59 connections. This suggests a strong integration between the default mode and limbic networks, which are both involved in emotional and self-related processing. In ADHD, there's often reduced integration between these networks, so a high value here might lean towards Control. The Unassigned-Vis connection is 47, which is quite high. The Unassigned re

Subjects: 100%|██████████| 768/768 [2:58:45<00:00, 13.97s/subj, done=768, err=0, ok=757, parse_fail=11] 


Finished this run in 178.8 min.
Total saved: 768/768  (ok=757, parse_fail=11, error=0)
All subjects done -> deepseek/community/deepseek_community_gt_36_10com_1.json


In [11]:
import json, os

VALID_LABELS = {"ADHD", "Control"}
FINAL_PATH = "deepseek/community/deepseek_community_gt_36_10com_1.json"

def print_stats(data):
    n = len(data)

    n_no_pred = sum(1 for r in data.values() if r.get("prediction") is None)
    malformed_entries = [
        (sid, r.get("prediction"))
        for sid, r in data.items()
        if r.get("prediction") is not None and r.get("prediction") not in VALID_LABELS
    ]
    n_malformed_pred = len(malformed_entries)

    # scorable = BOTH ground_truth and prediction are exactly "ADHD" or "Control"
    scorable = [
        r for r in data.values()
        if r.get("ground_truth") in VALID_LABELS and r.get("prediction") in VALID_LABELS
    ]
    n_scorable = len(scorable)

    n_gt_adhd_all = sum(1 for r in data.values() if r.get("ground_truth") == "ADHD")
    n_gt_control_all = sum(1 for r in data.values() if r.get("ground_truth") == "Control")

    n_pred_adhd = sum(1 for r in scorable if r["prediction"] == "ADHD")
    n_pred_control = sum(1 for r in scorable if r["prediction"] == "Control")

    n_correct = sum(1 for r in scorable if r["prediction"] == r["ground_truth"])

    # Per-class, denominator restricted to scorable subjects only
    scorable_gt_adhd = [r for r in scorable if r["ground_truth"] == "ADHD"]
    scorable_gt_control = [r for r in scorable if r["ground_truth"] == "Control"]
    n_adhd_correct = sum(1 for r in scorable_gt_adhd if r["prediction"] == "ADHD")
    n_control_correct = sum(1 for r in scorable_gt_control if r["prediction"] == "Control")

    print(f"Total subjects in file:              {n}")
    print(f"  -- no prediction (parse failure):  {n_no_pred}")
    print(f"  -- malformed prediction (neither ADHD nor Control exactly): {n_malformed_pred}")
    if malformed_entries:
        print(f"     malformed subject_ids and their prediction values:")
        for sid, val in malformed_entries:
            print(f"       {sid}: {val!r}")
    print(f"  -- scorable (valid ground_truth AND valid prediction): {n_scorable}")
    print()
    print(f"Ground truth (all {n} subjects) -- ADHD: {n_gt_adhd_all} | Control: {n_gt_control_all}")
    print(f"Prediction (scorable subjects only, n={n_scorable}) -- ADHD: {n_pred_adhd} | Control: {n_pred_control}")
    print()
    if n_scorable:
        print(f"Accuracy (scorable only): {n_correct}/{n_scorable} = {n_correct / n_scorable:.2%}")
    else:
        print("Accuracy: n/a (no scorable subjects)")

    print()
    print("Correctly predicted, by class (denominator = scorable subjects of that class only):")
    if scorable_gt_adhd:
        print(f"  ADHD    correctly predicted: {n_adhd_correct}/{len(scorable_gt_adhd)} = {n_adhd_correct / len(scorable_gt_adhd):.2%}")
    else:
        print("  ADHD    correctly predicted: n/a")
    if scorable_gt_control:
        print(f"  Control correctly predicted: {n_control_correct}/{len(scorable_gt_control)} = {n_control_correct / len(scorable_gt_control):.2%}")
    else:
        print("  Control correctly predicted: n/a")


if os.path.exists(FINAL_PATH):
    with open(FINAL_PATH) as f:
        data = json.load(f)
    print("RUN COMPLETE.")
    print_stats(data)
elif os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        data = json.load(f)
    print_stats(data)

else:
    print("No output file found yet -- nothing processed.")

print(FINAL_PATH)

RUN COMPLETE.
Total subjects in file:              768
  -- no prediction (parse failure):  11
  -- malformed prediction (neither ADHD nor Control exactly): 0
  -- scorable (valid ground_truth AND valid prediction): 757

Ground truth (all 768 subjects) -- ADHD: 280 | Control: 488
Prediction (scorable subjects only, n=757) -- ADHD: 367 | Control: 390

Accuracy (scorable only): 389/757 = 51.39%

Correctly predicted, by class (denominator = scorable subjects of that class only):
  ADHD    correctly predicted: 137/275 = 49.82%
  Control correctly predicted: 252/482 = 52.28%
deepseek/community/deepseek_community_gt_36_10com_1.json
